# Chapter 05: The First Search Baseline

Scan, indexed overlap and BM25 on a labelled tiny fixture. Recall@k, MRR and undefined no-positive cases.

In [ ]:
import sys
from pathlib import Path

# Resolve the repository root by walking parents from the current working directory.
ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

# Make this notebook's shared helper importable, then re-resolve through it.
NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)

## Evaluate on the development split

In [ ]:
import json
from document_identity import Collection
from lexical_index import InvertedIndex
from lexical_evaluation import evaluate

collection = Collection.load(ROOT / "examples" / "fixtures" / "part1-v1.json")
index = InvertedIndex.from_collection(collection, "baseline")
dev_queries = json.load(open(ROOT / "examples" / "fixtures" / "part1-development.json"))["queries"]

dev_result = evaluate(index, dev_queries, ["scan", "overlap", "bm25"], k1=1.2, b=0.75, k_values=(1, 3), rr_cutoff=3)
print(json.dumps(dev_result["aggregate"], indent=2))

assert dev_result["aggregate"]["scan"]["mean_recall_at"]["1"] == dev_result["aggregate"]["overlap"]["mean_recall_at"]["1"]
assert dev_result["aggregate"]["bm25"]["mean_recall_at"]["1"] > dev_result["aggregate"]["scan"]["mean_recall_at"]["1"]

## Evaluate on the held-out split

In [ ]:
heldout_queries = json.load(open(ROOT / "examples" / "fixtures" / "part1-heldout.json"))["queries"]
heldout_result = evaluate(index, heldout_queries, ["scan", "overlap", "bm25"], k1=1.2, b=0.75, k_values=(1, 3), rr_cutoff=3)
print(json.dumps(heldout_result["aggregate"], indent=2))

for method in ["scan", "overlap", "bm25"]:
    assert heldout_result["aggregate"][method]["mean_recall_at"]["1"] == heldout_result["aggregate"]["scan"]["mean_recall_at"]["1"]

## Query-level change on development data

In [ ]:
for obs in dev_result["queries"]:
    overlap_first = obs["methods"]["overlap"]["trace"]["selected"][0]["passage_id"] if obs["methods"]["overlap"]["trace"]["selected"] else None
    bm25_first = obs["methods"]["bm25"]["trace"]["selected"][0]["passage_id"] if obs["methods"]["bm25"]["trace"]["selected"] else None
    if overlap_first != bm25_first:
        print(obs["query_id"], overlap_first, "->", bm25_first)

## Undefined metric for no-positive query

In [ ]:
from lexical_evaluation import metrics

no_positive = metrics(["passage:does-not-matter"], {"passage:does-not-matter": 0})
print(no_positive)
assert no_positive["answerable"] is False
assert no_positive["recall_at"]["1"] is None
assert no_positive["recall_at"]["3"] is None
assert no_positive["rr_at"] is None
assert no_positive["returned_candidates"] is True

empty_result = metrics([], {"passage:does-not-matter": 0})
assert empty_result["returned_candidates"] is False

## Try it yourself
Inspect a per-query trace or change k_values. Keep the query split and judgements fixed.

In [ ]:
example = dev_result["queries"][0]
print(example["query_id"], example["query"])
print("overlap metrics:", example["methods"]["overlap"]["metrics"])

## Observed versus not established
Observed: aggregate and per-query lexical comparison on a tiny authored fixture. Not established: independent authorship, blind evaluation, latency, memory, truth, permission, admission, use or helpfulness.